# AHP Expert Questionnaire for Hub Prioritization

## Analytic Hierarchy Process (AHP) Scoring

This notebook guides you through the **Analytic Hierarchy Process (AHP)** to determine weights for the hub prioritization criteria. AHP is a structured decision-making methodology developed by Thomas Saaty that uses pairwise comparisons to derive criterion weights.

### How AHP Works

1. **Pairwise Comparisons**: You compare each pair of criteria and decide which is more important
2. **Saaty Scale**: Use a 1-9 scale to express the intensity of importance
3. **Weight Calculation**: Weights are calculated using the eigenvector method
4. **Consistency Check**: Your answers are validated for logical consistency (CR < 0.10)

### The 5 Scoring Criteria

| Criterion | Hebrew | Description |
|-----------|--------|-------------|
| **Activity Score** | ציון פעילות | 2050 passenger demand forecast (log-transformed) |
| **Service Score** | ציון שירות | Transit service quality (modes, frequencies, diversity) |
| **Location Score** | ציון מיקום | Geographic/strategic location importance |
| **Pop/Jobs Score** | ציון אוכלוסייה ותעסוקה | Population and employment in catchment area |
| **Terminal Score** | ציון מסוף | Bus terminal proximity and integration |

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, Layout, VBox, HBox, Label

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries loaded successfully!")

## The Saaty Scale

When comparing two criteria (A vs B), use the following scale:

| Value | Meaning | Description |
|-------|---------|-------------|
| **1** | Equal | Both criteria are equally important |
| **2** | Weak | Criterion A is slightly more important |
| **3** | Moderate | Criterion A is moderately more important |
| **4** | Moderate+ | Criterion A is moderately to strongly more important |
| **5** | Strong | Criterion A is strongly more important |
| **6** | Strong+ | Criterion A is strongly to very strongly more important |
| **7** | Very Strong | Criterion A is very strongly more important |
| **8** | Very Strong+ | Criterion A is very strongly to extremely more important |
| **9** | Extreme | Criterion A is extremely more important |

**Reciprocals**: If B is more important than A, use values **1/2, 1/3, ..., 1/9**

### Examples
- "Activity Score vs Service Score = 3" means Activity is **moderately** more important than Service
- "Activity Score vs Service Score = 1/5" (0.2) means Service is **strongly** more important than Activity

In [ ]:
# Define the criteria
CRITERIA = [
    'activity_score',
    'service_score', 
    'location_score',
    'pop_jobs_score',
    'terminal_score'
]

CRITERIA_LABELS = {
    'activity_score': 'Passenger Activity (ציון פעילות)',
    'service_score': 'Service & Modes (ציון שירות)',
    'location_score': 'Location (ציון מיקום)',
    'pop_jobs_score': 'Population & Jobs (אוכלוסייה ותעסוקה)',
    'terminal_score': 'Bus Terminal (ציון מסוף)'
}

CRITERIA_DESCRIPTIONS = {
    'activity_score': 'Forecasted passenger demand for 2050 (log-transformed to prevent mega-station dominance)',
    'service_score': 'Quality of transit service including mode types, frequencies, and multimodal diversity',
    'location_score': 'Strategic geographic importance (periphery vs. center, core vs. outer ring)',
    'pop_jobs_score': 'Population and employment density within walking distance (catchment area)',
    'terminal_score': 'Integration with bus network through proximity to bus terminals'
}

# Random Index values for consistency ratio (Saaty, 1980)
RANDOM_INDEX = {
    1: 0.00, 2: 0.00, 3: 0.58, 4: 0.90, 5: 1.12,
    6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49
}

# Saaty scale options for dropdown
SAATY_OPTIONS = [
    ('9 - Extremely more important', 9),
    ('8 - Very to extremely more important', 8),
    ('7 - Very strongly more important', 7),
    ('6 - Strongly to very strongly more important', 6),
    ('5 - Strongly more important', 5),
    ('4 - Moderately to strongly more important', 4),
    ('3 - Moderately more important', 3),
    ('2 - Slightly more important', 2),
    ('1 - Equal importance', 1),
    ('1/2 - Slightly less important', 0.5),
    ('1/3 - Moderately less important', 1/3),
    ('1/4 - Moderately to strongly less important', 0.25),
    ('1/5 - Strongly less important', 0.2),
    ('1/6 - Strongly to very strongly less important', 1/6),
    ('1/7 - Very strongly less important', 1/7),
    ('1/8 - Very to extremely less important', 0.125),
    ('1/9 - Extremely less important', 1/9),
]

print(f"Number of criteria: {len(CRITERIA)}")
print(f"Number of pairwise comparisons needed: {len(CRITERIA) * (len(CRITERIA) - 1) // 2}")

## Step 1: Enter Your Expert Information

In [ ]:
# Expert name input
expert_name_widget = widgets.Text(
    value='expert1',
    placeholder='Enter your name or ID',
    description='Expert Name:',
    style={'description_width': '120px'},
    layout=Layout(width='400px')
)

display(HTML("<h3>Enter your expert identifier:</h3>"))
display(expert_name_widget)
display(HTML("<p><i>This will be used to identify your responses in the output file.</i></p>"))

## Step 2: Pairwise Comparisons

For each pair of criteria, select how important the **first criterion** is compared to the **second criterion**.

- **Values > 1**: First criterion is MORE important
- **Value = 1**: Both are EQUALLY important  
- **Values < 1**: First criterion is LESS important (second is more important)

In [ ]:
# Generate all pairwise combinations
comparisons = []
for i in range(len(CRITERIA)):
    for j in range(i + 1, len(CRITERIA)):
        comparisons.append((CRITERIA[i], CRITERIA[j]))

# Create widgets for each comparison
comparison_widgets = {}

display(HTML("<h3>Pairwise Comparisons</h3>"))
display(HTML("<p>For each comparison, select how important the <b>first criterion</b> is relative to the <b>second criterion</b>.</p>"))
display(HTML("<hr>"))

for idx, (crit_a, crit_b) in enumerate(comparisons, 1):
    label_a = CRITERIA_LABELS[crit_a]
    label_b = CRITERIA_LABELS[crit_b]
    
    # Display comparison header
    display(HTML(f"<h4>Comparison {idx}/{len(comparisons)}</h4>"))
    display(HTML(f"<p><b>{label_a}</b> vs <b>{label_b}</b></p>"))
    display(HTML(f"<p><small><i>{crit_a}: {CRITERIA_DESCRIPTIONS[crit_a]}</i></small></p>"))
    display(HTML(f"<p><small><i>{crit_b}: {CRITERIA_DESCRIPTIONS[crit_b]}</i></small></p>"))
    
    # Create dropdown widget
    widget = widgets.Dropdown(
        options=SAATY_OPTIONS,
        value=1,  # Default to equal importance
        description=f'{label_a} is:',
        style={'description_width': 'initial'},
        layout=Layout(width='500px')
    )
    
    comparison_widgets[(crit_a, crit_b)] = widget
    display(widget)
    display(HTML("<hr>"))

## Step 3: Calculate Weights and Check Consistency

Run the cell below to:
1. Build the pairwise comparison matrix from your answers
2. Calculate priority weights using the eigenvector method
3. Check consistency ratio (should be < 0.10)

In [ ]:
def build_comparison_matrix(comparison_widgets, criteria):
    """Build pairwise comparison matrix from widget values."""
    n = len(criteria)
    matrix = np.eye(n)  # Diagonal is 1
    criteria_idx = {c: i for i, c in enumerate(criteria)}
    
    for (crit_a, crit_b), widget in comparison_widgets.items():
        i = criteria_idx[crit_a]
        j = criteria_idx[crit_b]
        value = widget.value
        
        matrix[i, j] = value
        matrix[j, i] = 1.0 / value  # Reciprocal
    
    return matrix


def calculate_priority_weights(matrix):
    """Calculate priority weights using eigenvector method."""
    eigenvalues, eigenvectors = np.linalg.eig(matrix)
    max_idx = np.argmax(eigenvalues.real)
    max_eigenvector = eigenvectors[:, max_idx].real
    weights = max_eigenvector / max_eigenvector.sum()
    return weights, eigenvalues.real[max_idx]


def calculate_consistency_ratio(matrix, weights, lambda_max):
    """Calculate Consistency Index (CI) and Consistency Ratio (CR)."""
    n = matrix.shape[0]
    ci = (lambda_max - n) / (n - 1) if n > 1 else 0
    ri = RANDOM_INDEX.get(n, 1.12)
    cr = ci / ri if ri > 0 else 0
    return ci, cr


# Build matrix and calculate
matrix = build_comparison_matrix(comparison_widgets, CRITERIA)
weights, lambda_max = calculate_priority_weights(matrix)
ci, cr = calculate_consistency_ratio(matrix, weights, lambda_max)

# Display results
display(HTML("<h3>Results</h3>"))

# Pairwise comparison matrix
display(HTML("<h4>Pairwise Comparison Matrix</h4>"))
matrix_df = pd.DataFrame(
    matrix,
    index=[CRITERIA_LABELS[c] for c in CRITERIA],
    columns=[CRITERIA_LABELS[c] for c in CRITERIA]
)
display(matrix_df.round(3))

# Priority weights
display(HTML("<h4>Priority Weights</h4>"))
weights_df = pd.DataFrame({
    'Criterion': [CRITERIA_LABELS[c] for c in CRITERIA],
    'Weight': weights,
    'Percentage': [f"{w*100:.1f}%" for w in weights]
})
display(weights_df)

# Consistency check
display(HTML("<h4>Consistency Check</h4>"))
is_consistent = cr < 0.10

consistency_html = f"""
<table style="border-collapse: collapse; width: 50%;">
    <tr><td style="border: 1px solid black; padding: 8px;"><b>λ_max (Principal Eigenvalue)</b></td>
        <td style="border: 1px solid black; padding: 8px;">{lambda_max:.4f}</td></tr>
    <tr><td style="border: 1px solid black; padding: 8px;"><b>Consistency Index (CI)</b></td>
        <td style="border: 1px solid black; padding: 8px;">{ci:.4f}</td></tr>
    <tr><td style="border: 1px solid black; padding: 8px;"><b>Random Index (RI)</b></td>
        <td style="border: 1px solid black; padding: 8px;">{RANDOM_INDEX[len(CRITERIA)]:.2f}</td></tr>
    <tr><td style="border: 1px solid black; padding: 8px;"><b>Consistency Ratio (CR)</b></td>
        <td style="border: 1px solid black; padding: 8px;">{cr:.4f}</td></tr>
    <tr><td style="border: 1px solid black; padding: 8px;"><b>Threshold</b></td>
        <td style="border: 1px solid black; padding: 8px;">0.10</td></tr>
    <tr><td style="border: 1px solid black; padding: 8px;"><b>Status</b></td>
        <td style="border: 1px solid black; padding: 8px; color: {'green' if is_consistent else 'red'};">
            {'✓ CONSISTENT' if is_consistent else '✗ INCONSISTENT - Please revise your answers'}</td></tr>
</table>
"""
display(HTML(consistency_html))

if not is_consistent:
    display(HTML("""
    <div style="background-color: #ffcccc; padding: 15px; border-radius: 5px; margin-top: 10px;">
        <h4 style="color: red;">⚠️ Inconsistent Judgments Detected</h4>
        <p>Your consistency ratio (CR = {:.4f}) exceeds the acceptable threshold of 0.10.</p>
        <p>This indicates some logical inconsistency in your pairwise comparisons.</p>
        <p><b>Recommendations:</b></p>
        <ul>
            <li>Review your comparisons for transitivity (if A > B and B > C, then A should be > C)</li>
            <li>Consider if any extreme values (7, 8, 9) are justified</li>
            <li>Go back to Step 2 and revise your answers</li>
            <li>Then re-run this cell to check again</li>
        </ul>
    </div>
    """.format(cr)))
else:
    display(HTML("""
    <div style="background-color: #ccffcc; padding: 15px; border-radius: 5px; margin-top: 10px;">
        <h4 style="color: green;">✓ Judgments are Consistent</h4>
        <p>Your consistency ratio (CR = {:.4f}) is within the acceptable threshold.</p>
        <p>You can proceed to export your weights in the next step.</p>
    </div>
    """.format(cr)))

# Store results for export
ahp_results = {
    'matrix': matrix,
    'weights': weights,
    'lambda_max': lambda_max,
    'ci': ci,
    'cr': cr,
    'is_consistent': is_consistent
}

## Step 4: Export to CSV

Run the cell below to export your pairwise comparisons to a CSV file that can be used with the AHP scoring module.

In [ ]:
def export_to_csv(expert_name, comparison_widgets, criteria, output_dir=None):
    """Export pairwise comparisons to CSV in the expected format."""
    
    # Build rows for CSV
    rows = []
    for (crit_a, crit_b), widget in comparison_widgets.items():
        rows.append({
            'expert': expert_name,
            'criterion_a': crit_a,
            'criterion_b': crit_b,
            'value': widget.value
        })
    
    df = pd.DataFrame(rows)
    
    # Determine output path
    if output_dir is None:
        output_dir = Path('../data')
    else:
        output_dir = Path(output_dir)
    
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Generate filename with timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f'ahp_expert_comparisons_{expert_name}_{timestamp}.csv'
    output_path = output_dir / filename
    
    # Save CSV
    df.to_csv(output_path, index=False)
    
    return output_path, df


def export_weights_summary(expert_name, criteria, weights, cr, output_dir=None):
    """Export a summary of weights for easy reference."""
    
    if output_dir is None:
        output_dir = Path('../data')
    else:
        output_dir = Path(output_dir)
    
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Generate filename
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f'ahp_weights_summary_{expert_name}_{timestamp}.csv'
    output_path = output_dir / filename
    
    # Create summary dataframe
    df = pd.DataFrame({
        'criterion': criteria,
        'weight': weights,
        'percentage': [f"{w*100:.2f}%" for w in weights]
    })
    df.loc[len(df)] = ['_consistency_ratio', cr, f"CR={cr:.4f}"]
    df.loc[len(df)] = ['_expert', expert_name, expert_name]
    
    df.to_csv(output_path, index=False)
    
    return output_path, df


# Get expert name
expert_name = expert_name_widget.value.strip()
if not expert_name:
    expert_name = 'expert1'

# Export files
display(HTML("<h3>Exporting Results</h3>"))

# Export pairwise comparisons
comparisons_path, comparisons_df = export_to_csv(
    expert_name, 
    comparison_widgets, 
    CRITERIA
)

display(HTML(f"<p>✓ Pairwise comparisons saved to: <code>{comparisons_path}</code></p>"))

# Export weights summary
weights_path, weights_summary_df = export_weights_summary(
    expert_name,
    CRITERIA,
    weights,
    cr
)

display(HTML(f"<p>✓ Weights summary saved to: <code>{weights_path}</code></p>"))

# Show exported data
display(HTML("<h4>Exported Pairwise Comparisons</h4>"))
display(comparisons_df)

display(HTML("<h4>Exported Weights Summary</h4>"))
display(weights_summary_df)

## Step 5: Append to Main Expert File (Optional)

If you want to add your comparisons to the main `ahp_expert_comparisons.csv` file (which may contain responses from multiple experts), run the cell below.

In [ ]:
def append_to_main_file(expert_name, comparison_widgets, main_file_path=None):
    """Append expert comparisons to the main AHP file."""
    
    if main_file_path is None:
        main_file_path = Path('../data/ahp_expert_comparisons.csv')
    else:
        main_file_path = Path(main_file_path)
    
    # Build new rows
    new_rows = []
    for (crit_a, crit_b), widget in comparison_widgets.items():
        new_rows.append({
            'expert': expert_name,
            'criterion_a': crit_a,
            'criterion_b': crit_b,
            'value': widget.value
        })
    
    new_df = pd.DataFrame(new_rows)
    
    # Check if main file exists
    if main_file_path.exists():
        existing_df = pd.read_csv(main_file_path)
        
        # Check if expert already exists
        if expert_name in existing_df['expert'].values:
            # Remove old entries for this expert
            existing_df = existing_df[existing_df['expert'] != expert_name]
            display(HTML(f"<p>⚠️ Replacing existing entries for expert '{expert_name}'</p>"))
        
        # Append new rows
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined_df = new_df
    
    # Save
    combined_df.to_csv(main_file_path, index=False)
    
    # Count experts
    n_experts = combined_df['expert'].nunique()
    
    return main_file_path, combined_df, n_experts


# Confirm before appending
display(HTML("<h3>Append to Main Expert File</h3>"))
display(HTML(f"<p>Expert name: <b>{expert_name}</b></p>"))
display(HTML(f"<p>Target file: <code>data/ahp_expert_comparisons.csv</code></p>"))

append_button = widgets.Button(
    description='Append to Main File',
    button_style='success',
    icon='check'
)

output_area = widgets.Output()

def on_append_click(b):
    with output_area:
        output_area.clear_output()
        main_path, combined_df, n_experts = append_to_main_file(
            expert_name,
            comparison_widgets
        )
        display(HTML(f"<p style='color: green;'>✓ Successfully appended to {main_path}</p>"))
        display(HTML(f"<p>Total experts in file: <b>{n_experts}</b></p>"))
        display(HTML("<h4>Current File Contents</h4>"))
        display(combined_df)

append_button.on_click(on_append_click)

display(append_button)
display(output_area)

## Visualization: Weight Distribution

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart
    ax1 = axes[0]
    colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#f39c12']
    bars = ax1.barh(
        [CRITERIA_LABELS[c] for c in CRITERIA],
        weights,
        color=colors
    )
    ax1.set_xlabel('Weight')
    ax1.set_title('Criterion Weights (Bar Chart)')
    ax1.set_xlim(0, max(weights) * 1.2)
    
    # Add value labels
    for bar, w in zip(bars, weights):
        ax1.text(w + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{w:.3f} ({w*100:.1f}%)', va='center')
    
    # Pie chart
    ax2 = axes[1]
    ax2.pie(
        weights,
        labels=[CRITERIA_LABELS[c].split(' (')[0] for c in CRITERIA],
        autopct='%1.1f%%',
        colors=colors,
        startangle=90
    )
    ax2.set_title('Criterion Weights (Pie Chart)')
    
    plt.tight_layout()
    plt.show()
    
    # Save figure
    fig_path = Path('../data') / f'ahp_weights_{expert_name}.png'
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    display(HTML(f"<p>✓ Figure saved to: <code>{fig_path}</code></p>"))
    
except ImportError:
    display(HTML("<p><i>matplotlib not available - skipping visualization</i></p>"))

---

## Summary

This notebook has helped you:

1. **Learn about AHP** - The Analytic Hierarchy Process and Saaty scale
2. **Make pairwise comparisons** - Compare all 5 criteria pairs
3. **Calculate weights** - Derive priority weights using eigenvector method
4. **Check consistency** - Validate your judgments (CR < 0.10)
5. **Export results** - Save to CSV for use in hub scoring

### Next Steps

To use your weights in the hub scoring pipeline:

1. Ensure `AHP_ENABLED = True` in `src/config.py`
2. Place your exported CSV in `data/ahp_expert_comparisons.csv`
3. Run the scoring pipeline - it will automatically use AHP weights

### Multiple Experts

For multi-expert analysis:
- Have each expert run this notebook with a unique expert name
- Use the "Append to Main File" feature to combine responses
- The scoring module will aggregate weights using geometric mean

---

**References:**
- Saaty, T.L. (1980). *The Analytic Hierarchy Process*. McGraw-Hill.
- Saaty, T.L. (2008). Decision making with the analytic hierarchy process. *IJSSCI*, 1(1), 83-98.